# Smart Waste Collection Simulation
# Scenario & Sensitivity Analysis
#
# Objective:
# Compare alternative collection decision thresholds using the
# validated Monte Carlo simulation model developed by Member 5,
# and test how sensitive the results are to changes in the
# assumed accumulation behaviour.

In [ ]:
# ---------------------------------------------------------
# Step 1: Load and verify 
# ---------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load Member 5 outputs
base_results = pd.read_csv(
    "simulation_base_results.csv"
)

base_run_summary = pd.read_csv(
    "simulation_base_run_summary.csv"
)

base_summary = pd.read_csv(
    "simulation_base_summary.csv"
)

# Load Member 4 random-variable inputs
rv1 = pd.read_csv(
    "simulation_RV1_accumulation_occurrence.csv"
)

rv2 = pd.read_csv(
    "simulation_RV2_increase_magnitude.csv"
)


print("Base detailed results:")
print(base_results.shape)

print("\nBase run summary:")
print(base_run_summary.shape)

print("\nBase overall summary:")
print(base_summary.shape)

print("\nRV1:")
print(rv1.shape)

print("\nRV2:")
print(rv2.shape)


print("\n-----------------------------------")
print("Base simulation checks")
print("-----------------------------------")

print(
    "Number of simulation runs:",
    base_results["run"].nunique()
)

print(
    "Days per run:",
    base_results.groupby("run")["day"].count().unique()
)

print("\nBase summary:")
print(base_summary.to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# Step 2: Load validated simulation engine
# ---------------------------------------------------------

def sample_accumulation_event(current_fullness, rng):

    row = rv1[
        rv1["previous_fullness"] == current_fullness
    ].iloc[0]

    event = rng.choice(
        ["Increase", "No Change"],
        p=[
            row["Increase"],
            row["No Change"]
        ]
    )

    return event


def sample_increase_amount(current_fullness, rng):

    row = rv2[
        rv2["previous_fullness"] == current_fullness
    ].iloc[0]

    increase_amounts = [2, 4, 6, 8, 10]

    probabilities = [
        row["increase_2"],
        row["increase_4"],
        row["increase_6"],
        row["increase_8"],
        row["increase_10"]
    ]

    increase = rng.choice(
        increase_amounts,
        p=probabilities
    )

    return int(increase)


def simulate_accumulation_day(current_fullness, rng):

    event = sample_accumulation_event(
        current_fullness,
        rng
    )

    if event == "No Change":

        increase_amount = 0
        new_fullness = current_fullness

    else:

        increase_amount = sample_increase_amount(
            current_fullness,
            rng
        )

        new_fullness = (
            current_fullness + increase_amount
        )

    return {
        "start_fullness": current_fullness,
        "accumulation_event": str(event),
        "increase_amount": increase_amount,
        "new_fullness": new_fullness
    }


def apply_collection_rule(
    pre_collection_fullness,
    collection_threshold
):

    if pre_collection_fullness >= collection_threshold:

        collected = True
        end_fullness = 0

    else:

        collected = False
        end_fullness = pre_collection_fullness

    return {
        "collected": collected,
        "end_fullness": end_fullness
    }


def simulate_one_day(
    current_fullness,
    collection_threshold,
    rng
):

    accumulation = simulate_accumulation_day(
        current_fullness,
        rng
    )

    pre_collection_fullness = (
        accumulation["new_fullness"]
    )

    collection = apply_collection_rule(
        pre_collection_fullness,
        collection_threshold
    )

    return {
        "start_fullness":
            current_fullness,

        "accumulation_event":
            accumulation["accumulation_event"],

        "increase_amount":
            accumulation["increase_amount"],

        "pre_collection_fullness":
            pre_collection_fullness,

        "collected":
            collection["collected"],

        "end_fullness":
            collection["end_fullness"]
    }


def simulate_bin(
    number_of_days,
    starting_fullness,
    collection_threshold,
    seed
):

    rng = np.random.default_rng(seed)

    current_fullness = starting_fullness

    records = []

    for day in range(1, number_of_days + 1):

        result = simulate_one_day(
            current_fullness,
            collection_threshold,
            rng
        )

        result["day"] = day

        records.append(result)

        current_fullness = (
            result["end_fullness"]
        )

    return pd.DataFrame(records)


def run_monte_carlo(
    number_of_runs,
    number_of_days,
    starting_fullness,
    collection_threshold,
    base_seed
):

    all_results = []

    for run in range(1, number_of_runs + 1):

        run_seed = base_seed + run

        simulation = simulate_bin(
            number_of_days=number_of_days,
            starting_fullness=starting_fullness,
            collection_threshold=collection_threshold,
            seed=run_seed
        )

        simulation["run"] = run

        all_results.append(simulation)

    return pd.concat(
        all_results,
        ignore_index=True
    )


print("Simulation engine loaded successfully.")

In [ ]:
# ---------------------------------------------------------
# Step 3: Reproduce base case
# ---------------------------------------------------------

member6_base_check = run_monte_carlo(
    number_of_runs=1000,
    number_of_days=365,
    starting_fullness=0,
    collection_threshold=8,
    base_seed=42
)

# Make sure column order matches
member6_base_check = member6_base_check[
    base_results.columns
]

# Compare with base results
identical = member6_base_check.equals(
    base_results
)

print(
    "Member 6 reproduction identical to base:",
    identical
)

print(
    "Member 6 rows:",
    len(member6_base_check)
)

print(
    "Base rows:",
    len(base_results)
)

In [ ]:
# ---------------------------------------------------------
# Step 4: Run collection-threshold scenarios
# ---------------------------------------------------------

scenario_6 = run_monte_carlo(
    number_of_runs=1000,
    number_of_days=365,
    starting_fullness=0,
    collection_threshold=6,
    base_seed=42
)

scenario_8 = run_monte_carlo(
    number_of_runs=1000,
    number_of_days=365,
    starting_fullness=0,
    collection_threshold=8,
    base_seed=42
)

scenario_10 = run_monte_carlo(
    number_of_runs=1000,
    number_of_days=365,
    starting_fullness=0,
    collection_threshold=10,
    base_seed=42
)


print("Threshold 6:", scenario_6.shape)
print("Threshold 8:", scenario_8.shape)
print("Threshold 10:", scenario_10.shape)

print("\nUnique runs:")
print("Threshold 6:", scenario_6["run"].nunique())
print("Threshold 8:", scenario_8["run"].nunique())
print("Threshold 10:", scenario_10["run"].nunique())

print("\nDays per run:")
print(
    "Threshold 6:",
    scenario_6.groupby("run")["day"].count().unique()
)

print(
    "Threshold 8:",
    scenario_8.groupby("run")["day"].count().unique()
)

print(
    "Threshold 10:",
    scenario_10.groupby("run")["day"].count().unique()
)

In [ ]:
# ---------------------------------------------------------
# Step 5: Calculate scenario KPIs
# ---------------------------------------------------------

def calculate_scenario_kpis(results, threshold):

    summary = (
        results.groupby("run")
        .agg(
            collections=("collected", "sum"),

            days_at_state_10=(
                "pre_collection_fullness",
                lambda x: (x == 10).sum()
            ),

            high_fullness_days=(
                "pre_collection_fullness",
                lambda x: x.isin([8, 10]).sum()
            ),

            average_fullness=(
                "pre_collection_fullness",
                "mean"
            )
        )
        .reset_index()
    )

    # Average fullness when collection occurred
    collection_fullness = (
        results[results["collected"] == True]
        .groupby("run")["pre_collection_fullness"]
        .mean()
        .rename("average_collection_fullness")
        .reset_index()
    )

    summary = summary.merge(
        collection_fullness,
        on="run",
        how="left"
    )

    # Percentage of simulated days at high fullness
    summary["high_fullness_percentage"] = (
        summary["high_fullness_days"] / 365 * 100
    )

    # Record scenario
    summary["collection_threshold"] = threshold

    return summary


summary_6 = calculate_scenario_kpis(
    scenario_6,
    6
)

summary_8 = calculate_scenario_kpis(
    scenario_8,
    8
)

summary_10 = calculate_scenario_kpis(
    scenario_10,
    10
)


print("Threshold 6:")
print(summary_6.describe().round(3))

print("\nThreshold 8:")
print(summary_8.describe().round(3))

print("\nThreshold 10:")
print(summary_10.describe().round(3))

In [ ]:
# ---------------------------------------------------------
# Step 6: Create scenario comparison table
# ---------------------------------------------------------

all_scenario_summaries = pd.concat(
    [summary_6, summary_8, summary_10],
    ignore_index=True
)

scenario_comparison = (
    all_scenario_summaries
    .groupby("collection_threshold")
    .agg(
        mean_collections=("collections", "mean"),
        std_collections=("collections", "std"),
        median_collections=("collections", "median"),
        p05_collections=("collections", lambda x: x.quantile(0.05)),
        p95_collections=("collections", lambda x: x.quantile(0.95)),

        mean_days_state_10=("days_at_state_10", "mean"),

        mean_fullness=("average_fullness", "mean"),

        mean_collection_fullness=(
            "average_collection_fullness",
            "mean"
        ),

        mean_high_fullness_pct=(
            "high_fullness_percentage",
            "mean"
        )
    )
    .reset_index()
)

print(
    scenario_comparison.round(3).to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# Step 7: Plot annual collections by threshold
# ---------------------------------------------------------

plot_data = [
    summary_6["collections"],
    summary_8["collections"],
    summary_10["collections"]
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    plot_data,
    tick_labels=[
        "Threshold 6",
        "Threshold 8",
        "Threshold 10"
    ]
)

plt.title(
    "Annual Collections Across 1,000 Simulated Years"
)

plt.xlabel(
    "Collection Threshold"
)

plt.ylabel(
    "Collections per Simulated Year"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ---------------------------------------------------------
# Step 8: Plot high-fullness exposure by threshold
# ---------------------------------------------------------

plot_data = [
    summary_6["high_fullness_percentage"],
    summary_8["high_fullness_percentage"],
    summary_10["high_fullness_percentage"]
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    plot_data,
    tick_labels=[
        "Threshold 6",
        "Threshold 8",
        "Threshold 10"
    ]
)

plt.title(
    "High-Fullness Exposure Across 1,000 Simulated Years"
)

plt.xlabel(
    "Collection Threshold"
)

plt.ylabel(
    "Days at State 8 or 10 (%)"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ---------------------------------------------------------
# Step 9: Create accumulation sensitivity RV1 tables
# ---------------------------------------------------------

def create_adjusted_rv1(original_rv1, multiplier):

    adjusted = original_rv1.copy()

    adjusted["Increase"] = (
        adjusted["Increase"] * multiplier
    ).clip(0, 1)

    adjusted["No Change"] = (
        1 - adjusted["Increase"]
    )

    return adjusted


rv1_lower = create_adjusted_rv1(
    rv1,
    0.80
)

rv1_base = create_adjusted_rv1(
    rv1,
    1.00
)

rv1_higher = create_adjusted_rv1(
    rv1,
    1.20
)


print("LOWER ACCUMULATION")
print(
    rv1_lower[
        ["previous_fullness", "Increase", "No Change"]
    ].round(4).to_string(index=False)
)

print("\nBASE ACCUMULATION")
print(
    rv1_base[
        ["previous_fullness", "Increase", "No Change"]
    ].round(4).to_string(index=False)
)

print("\nHIGHER ACCUMULATION")
print(
    rv1_higher[
        ["previous_fullness", "Increase", "No Change"]
    ].round(4).to_string(index=False)
)


print("\nProbability checks:")

print(
    "Lower:",
    (
        rv1_lower["Increase"]
        + rv1_lower["No Change"]
    ).round(10).unique()
)

print(
    "Base:",
    (
        rv1_base["Increase"]
        + rv1_base["No Change"]
    ).round(10).unique()
)

print(
    "Higher:",
    (
        rv1_higher["Increase"]
        + rv1_higher["No Change"]
    ).round(10).unique()
)

In [ ]:
# ---------------------------------------------------------
# Step 10: Create sensitivity-enabled simulation functions
# ---------------------------------------------------------

def sample_accumulation_event_sensitivity(
    current_fullness,
    rng,
    rv1_table
):

    row = rv1_table[
        rv1_table["previous_fullness"] == current_fullness
    ].iloc[0]

    event = rng.choice(
        ["Increase", "No Change"],
        p=[
            row["Increase"],
            row["No Change"]
        ]
    )

    return event


def simulate_accumulation_day_sensitivity(
    current_fullness,
    rng,
    rv1_table
):

    event = sample_accumulation_event_sensitivity(
        current_fullness,
        rng,
        rv1_table
    )

    if event == "No Change":

        increase_amount = 0
        new_fullness = current_fullness

    else:

        # RV2 remains unchanged
        increase_amount = sample_increase_amount(
            current_fullness,
            rng
        )

        new_fullness = (
            current_fullness + increase_amount
        )

    return {
        "start_fullness": current_fullness,
        "accumulation_event": str(event),
        "increase_amount": increase_amount,
        "new_fullness": new_fullness
    }


def simulate_one_day_sensitivity(
    current_fullness,
    collection_threshold,
    rng,
    rv1_table
):

    accumulation = simulate_accumulation_day_sensitivity(
        current_fullness,
        rng,
        rv1_table
    )

    pre_collection_fullness = (
        accumulation["new_fullness"]
    )

    collection = apply_collection_rule(
        pre_collection_fullness,
        collection_threshold
    )

    return {
        "start_fullness":
            current_fullness,

        "accumulation_event":
            accumulation["accumulation_event"],

        "increase_amount":
            accumulation["increase_amount"],

        "pre_collection_fullness":
            pre_collection_fullness,

        "collected":
            collection["collected"],

        "end_fullness":
            collection["end_fullness"]
    }


def simulate_bin_sensitivity(
    number_of_days,
    starting_fullness,
    collection_threshold,
    seed,
    rv1_table
):

    rng = np.random.default_rng(seed)

    current_fullness = starting_fullness
    records = []

    for day in range(1, number_of_days + 1):

        result = simulate_one_day_sensitivity(
            current_fullness,
            collection_threshold,
            rng,
            rv1_table
        )

        result["day"] = day
        records.append(result)

        current_fullness = result["end_fullness"]

    return pd.DataFrame(records)


def run_monte_carlo_sensitivity(
    number_of_runs,
    number_of_days,
    starting_fullness,
    collection_threshold,
    base_seed,
    rv1_table
):

    all_results = []

    for run in range(1, number_of_runs + 1):

        run_seed = base_seed + run

        simulation = simulate_bin_sensitivity(
            number_of_days=number_of_days,
            starting_fullness=starting_fullness,
            collection_threshold=collection_threshold,
            seed=run_seed,
            rv1_table=rv1_table
        )

        simulation["run"] = run

        all_results.append(simulation)

    return pd.concat(
        all_results,
        ignore_index=True
    )


print("Sensitivity-enabled simulation engine loaded successfully.")

In [ ]:
# ---------------------------------------------------------
# Step 11: Validate sensitivity-enabled simulation engine
# ---------------------------------------------------------

sensitivity_base_check = run_monte_carlo_sensitivity(
    number_of_runs=1000,
    number_of_days=365,
    starting_fullness=0,
    collection_threshold=8,
    base_seed=42,
    rv1_table=rv1_base
)

# Match column order with original base results
sensitivity_base_check = sensitivity_base_check[
    base_results.columns
]

# Compare results
identical = sensitivity_base_check.equals(
    base_results
)

print(
    "Sensitivity engine reproduces original base:",
    identical
)

print(
    "Sensitivity engine rows:",
    len(sensitivity_base_check)
)

print(
    "Original base rows:",
    len(base_results)
)

In [ ]:
# ---------------------------------------------------------
# Step 12: Run full sensitivity experiment
# ---------------------------------------------------------

accumulation_scenarios = {
    "Lower": rv1_lower,
    "Base": rv1_base,
    "Higher": rv1_higher
}

collection_thresholds = [
    6,
    8,
    10
]

sensitivity_results = {}


for accumulation_name, rv1_table in accumulation_scenarios.items():

    for threshold in collection_thresholds:

        print(
            f"Running: {accumulation_name} accumulation, "
            f"Threshold {threshold}"
        )

        result = run_monte_carlo_sensitivity(
            number_of_runs=1000,
            number_of_days=365,
            starting_fullness=0,
            collection_threshold=threshold,
            base_seed=42,
            rv1_table=rv1_table
        )

        sensitivity_results[
            (accumulation_name, threshold)
        ] = result


print("\nSensitivity experiments completed.")

print(
    "Number of experiments:",
    len(sensitivity_results)
)

print("\nRows in each experiment:")

for key, result in sensitivity_results.items():

    print(
        key,
        result.shape
    )

In [ ]:
# ---------------------------------------------------------
# Step 13: Calculate KPIs for sensitivity experiments
# ---------------------------------------------------------

sensitivity_summaries = []

for (
    accumulation_name,
    threshold
), result in sensitivity_results.items():

    summary = calculate_scenario_kpis(
        result,
        threshold
    )

    summary["accumulation_level"] = (
        accumulation_name
    )

    sensitivity_summaries.append(
        summary
    )


# Combine all 9 experiments
sensitivity_run_summary = pd.concat(
    sensitivity_summaries,
    ignore_index=True
)


# Create compact comparison table
sensitivity_comparison = (
    sensitivity_run_summary
    .groupby(
        [
            "accumulation_level",
            "collection_threshold"
        ]
    )
    .agg(
        mean_collections=(
            "collections",
            "mean"
        ),

        mean_days_state_10=(
            "days_at_state_10",
            "mean"
        ),

        mean_fullness=(
            "average_fullness",
            "mean"
        ),

        mean_collection_fullness=(
            "average_collection_fullness",
            "mean"
        ),

        mean_high_fullness_pct=(
            "high_fullness_percentage",
            "mean"
        )
    )
    .reset_index()
)


# Put accumulation levels in logical order
sensitivity_comparison[
    "accumulation_level"
] = pd.Categorical(
    sensitivity_comparison[
        "accumulation_level"
    ],
    categories=[
        "Lower",
        "Base",
        "Higher"
    ],
    ordered=True
)


sensitivity_comparison = (
    sensitivity_comparison
    .sort_values(
        [
            "accumulation_level",
            "collection_threshold"
        ]
    )
)


print(
    sensitivity_comparison
    .round(3)
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# Step 14: Plot sensitivity of annual collections
# ---------------------------------------------------------

plt.figure(figsize=(8, 5))

for threshold in [6, 8, 10]:

    data = sensitivity_comparison[
        sensitivity_comparison[
            "collection_threshold"
        ] == threshold
    ]

    plt.plot(
        data["accumulation_level"].astype(str),
        data["mean_collections"],
        marker="o",
        label=f"Threshold {threshold}"
    )


plt.title(
    "Sensitivity of Annual Collections to Accumulation"
)

plt.xlabel(
    "Accumulation Assumption"
)

plt.ylabel(
    "Mean Collections per Simulated Year"
)

plt.legend(
    title="Collection Rule"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ---------------------------------------------------------
# Step 15: Plot sensitivity of high-fullness exposure
# ---------------------------------------------------------

plt.figure(figsize=(8, 5))

for threshold in [6, 8, 10]:

    data = sensitivity_comparison[
        sensitivity_comparison[
            "collection_threshold"
        ] == threshold
    ]

    plt.plot(
        data["accumulation_level"].astype(str),
        data["mean_high_fullness_pct"],
        marker="o",
        label=f"Threshold {threshold}"
    )


plt.title(
    "Sensitivity of High-Fullness Exposure to Accumulation"
)

plt.xlabel(
    "Accumulation Assumption"
)

plt.ylabel(
    "Mean Days at State 8 or 10 (%)"
)

plt.legend(
    title="Collection Rule"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ---------------------------------------------------------
# Step 16: Validate all sensitivity experiments
# ---------------------------------------------------------

valid_states = {0, 2, 4, 6, 8, 10}

validation_records = []


for (
    accumulation_name,
    threshold
), result in sensitivity_results.items():

    # Basic structure
    correct_runs = (
        result["run"].nunique() == 1000
    )

    correct_days = (
        result.groupby("run")["day"]
        .count()
        .eq(365)
        .all()
    )

    # Valid fullness states
    valid_start_states = (
        set(result["start_fullness"].unique())
        .issubset(valid_states)
    )

    valid_pre_collection_states = (
        set(result["pre_collection_fullness"].unique())
        .issubset(valid_states)
    )

    valid_end_states = (
        set(result["end_fullness"].unique())
        .issubset(valid_states)
    )

    # No value above observed maximum state
    no_fullness_above_10 = (
        result["pre_collection_fullness"].max()
        <= 10
    )

    # Collection must reset bin to zero
    collection_resets_zero = (
        result.loc[
            result["collected"] == True,
            "end_fullness"
        ]
        .eq(0)
        .all()
    )

    # Daily continuity:
    # today's end fullness should equal
    # tomorrow's starting fullness
    next_start = (
        result.groupby("run")[
            "start_fullness"
        ].shift(-1)
    )

    continuity_mask = (
        result.groupby("run")["day"]
        .transform("max")
        != result["day"]
    )

    continuity_valid = (
        result.loc[
            continuity_mask,
            "end_fullness"
        ]
        .reset_index(drop=True)
        .equals(
            next_start.loc[
                continuity_mask
            ].reset_index(drop=True)
        )
    )

    validation_records.append({
        "accumulation_level":
            accumulation_name,

        "collection_threshold":
            threshold,

        "correct_runs":
            correct_runs,

        "correct_days":
            correct_days,

        "valid_start_states":
            valid_start_states,

        "valid_pre_collection_states":
            valid_pre_collection_states,

        "valid_end_states":
            valid_end_states,

        "no_fullness_above_10":
            no_fullness_above_10,

        "collection_resets_zero":
            collection_resets_zero,

        "continuity_valid":
            continuity_valid
    })


sensitivity_validation = pd.DataFrame(
    validation_records
)

print(
    sensitivity_validation
    .to_string(index=False)
)

print(
    "\nAll validation checks passed:",
    sensitivity_validation[
        sensitivity_validation
        .select_dtypes(include="bool")
        .columns
    ].all().all()
)

In [ ]:
# ---------------------------------------------------------
# Step 16A: Diagnose daily continuity
# ---------------------------------------------------------

continuity_records = []

for (
    accumulation_name,
    threshold
), result in sensitivity_results.items():

    # Sort explicitly before checking continuity
    check_df = result.sort_values(
        ["run", "day"]
    ).copy()

    # Previous day's ending fullness
    check_df["previous_end_fullness"] = (
        check_df.groupby("run")[
            "end_fullness"
        ].shift(1)
    )

    # Exclude Day 1 because it has no previous day
    continuity_rows = (
        check_df["day"] > 1
    )

    # Identify actual mismatches
    errors = check_df[
        continuity_rows
        & (
            check_df["start_fullness"]
            != check_df["previous_end_fullness"]
        )
    ]

    continuity_records.append({
        "accumulation_level":
            accumulation_name,

        "collection_threshold":
            threshold,

        "continuity_errors":
            len(errors),

        "continuity_valid":
            len(errors) == 0
    })


continuity_check = pd.DataFrame(
    continuity_records
)

print(
    continuity_check.to_string(
        index=False
    )
)

print(
    "\nAll continuity checks passed:",
    continuity_check[
        "continuity_valid"
    ].all()
)

In [ ]:
# ---------------------------------------------------------
# Step 17: Export Member 6 results
# ---------------------------------------------------------

# Scenario-level run results
all_scenario_summaries.to_csv(
    "scenario_run_summary.csv",
    index=False
)

# Compact scenario comparison
scenario_comparison.to_csv(
    "scenario_comparison.csv",
    index=False
)

# Sensitivity run-level results
sensitivity_run_summary.to_csv(
    "sensitivity_run_summary.csv",
    index=False
)

# Compact sensitivity comparison
sensitivity_comparison.to_csv(
    "sensitivity_comparison.csv",
    index=False
)


print("Files exported successfully:")

print(
    "1. scenario_run_summary.csv",
    all_scenario_summaries.shape
)

print(
    "2. scenario_comparison.csv",
    scenario_comparison.shape
)

print(
    "3. sensitivity_run_summary.csv",
    sensitivity_run_summary.shape
)

print(
    "4. sensitivity_comparison.csv",
    sensitivity_comparison.shape
)